# Oil Basket Consolidated Index (Colab) — robust version

This notebook is a **single-file**, free pipeline for Google Colab that:
1. Downloads up to **60 days** history for the requested contracts.
2. Collects the latest **20 minutes** every 15 minutes (or manually), deduplicating rows.
3. Produces an equal-weight consolidated index return over:
   `1w, 2d, 12h, 6h, 2h, 1h, 30m, 15m, 5m`.
4. Uses as-of carry-forward prices when exchanges are closed (including pre/post where available).

> Important: Some requested contracts are not reliably exposed by Yahoo Finance. This notebook avoids false positives (e.g., unrelated equities) and will keep unresolved contracts visible instead of silently using wrong symbols.

In [ ]:
# @title 1) Install dependencies
!pip -q install yfinance pandas numpy pytz

In [ ]:
# @title 2) Imports + config
import os
import time
import logging
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 80)

# Reduce noisy yfinance warnings in notebook output.
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

BASE_DIR = '/content/oil_index_data' if os.path.exists('/content') else './oil_index_data'
os.makedirs(BASE_DIR, exist_ok=True)

HISTORY_DIR = os.path.join(BASE_DIR, 'history_60d')
os.makedirs(HISTORY_DIR, exist_ok=True)

INTRADAY_FILE = os.path.join(BASE_DIR, 'intraday_last20m_dedup.csv')
METADATA_FILE = os.path.join(BASE_DIR, 'resolved_symbols.csv')
RETURNS_FILE = os.path.join(BASE_DIR, 'consolidated_index_returns.csv')

# Resolution mode:
# - 'maximize_coverage': accepts candidates with valid price history, marks confidence.
# - 'strict': requires futures/index-like metadata match.
RESOLUTION_MODE = 'maximize_coverage'

# Weighting behavior:
# requirement asks equal weight of listed contracts; default keeps duplicate symbols as separate contracts.
COUNT_DUPLICATE_SYMBOLS_MULTIPLE_TIMES = True

HORIZONS = {
    '1w': pd.Timedelta(days=7),
    '2d': pd.Timedelta(days=2),
    '12h': pd.Timedelta(hours=12),
    '6h': pd.Timedelta(hours=6),
    '2h': pd.Timedelta(hours=2),
    '1h': pd.Timedelta(hours=1),
    '30m': pd.Timedelta(minutes=30),
    '15m': pd.Timedelta(minutes=15),
    '5m': pd.Timedelta(minutes=5),
}

# Requested contracts + candidate symbols (editable)
CONTRACT_SPECS = {
    'CL — NYMEX WTI (Light Sweet Crude Oil)': {
        'code': 'CL',
        'candidates': ['CL=F'],
        'keywords': ['wti', 'crude', 'light sweet', 'nymex'],
    },
    'B — ICE Brent Crude': {
        'code': 'B',
        'candidates': ['BZ=F'],
        'keywords': ['brent', 'crude', 'ice'],
    },
    'BZ — NYMEX Brent Last Day Financial': {
        'code': 'BZ',
        'candidates': ['BZ=F'],
        'keywords': ['brent', 'crude', 'last day'],
    },
    'DBI — ICE Dubai 1st Line': {
        'code': 'DBI',
        'candidates': ['DBI=F', 'DBI.F'],
        'keywords': ['dubai', 'crude', 'ice'],
    },
    'DCB — NYMEX Dubai Crude Oil (Platts) Calendar Swap': {
        'code': 'DCB',
        'candidates': ['DCB=F', 'DCB.F'],
        'keywords': ['dubai', 'platts', 'swap', 'crude'],
    },
    'OQD — DME Oman Crude Oil Futures': {
        'code': 'OQD',
        'candidates': ['OQD=F', 'OQD.F'],
        'keywords': ['oman', 'dme', 'crude'],
    },
    'ADM — ICE Futures Abu Dhabi Murban Crude Oil Futures': {
        'code': 'ADM',
        'candidates': ['ADM=F', 'ADM.F'],
        'keywords': ['murban', 'abu dhabi', 'crude', 'ice'],
    },
    'HOU — ICE Midland WTI (Houston)': {
        'code': 'HOU',
        'candidates': ['HOU=F', 'HOU.F'],
        'keywords': ['midland', 'houston', 'wti', 'ice'],
    },
    'TMW — ICE Western Canadian Select (WCS) 1A Index': {
        'code': 'TMW',
        'candidates': ['TMW=F', 'TMW.F'],
        'keywords': ['western canadian', 'wcs', 'index', 'ice'],
    },
    'ARM — ICE Argus Mars': {
        'code': 'ARM',
        'candidates': ['ARM=F', 'ARM.F'],
        'keywords': ['argus', 'mars', 'crude', 'ice'],
    },
}

print('BASE_DIR:', BASE_DIR)


In [ ]:
# @title 3) Helper functions
def _standardize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [c[0] if isinstance(c, tuple) else c for c in x.columns]
    cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    keep = [c for c in cols if c in x.columns]
    x = x[keep].copy()
    x.index = pd.to_datetime(x.index, utc=True, errors='coerce')
    x = x[~x.index.isna()].sort_index()
    x['Px'] = x['Adj Close'] if 'Adj Close' in x.columns else x['Close']
    x = x.dropna(subset=['Px'])
    return x

def _get_symbol_info(symbol: str) -> dict:
    try:
        t = yf.Ticker(symbol)
        info = t.info if t.info is not None else {}
        return info if isinstance(info, dict) else {}
    except Exception:
        return {}

def _looks_like_target_instrument(symbol: str, info: dict, keywords: list[str]) -> tuple[bool, str]:
    quote_type = str(info.get('quoteType', '')).upper()
    text = ' '.join([
        str(info.get('shortName', '')),
        str(info.get('longName', '')),
        str(info.get('symbol', '')),
        str(info.get('exchange', '')),
    ]).lower()

    if symbol.endswith('=F'):
        return True, 'high_confidence_futures_symbol'

    keyword_match = any(k.lower() in text for k in keywords) if keywords else False
    if quote_type in {'FUTURE', 'INDEX'} and keyword_match:
        return True, 'high_confidence_metadata_match'

    if symbol.endswith('.F'):
        # coverage mode can accept .F candidates that have data, but mark as low confidence
        return (RESOLUTION_MODE == 'maximize_coverage'), 'low_confidence_dotF_fallback'

    return False, 'rejected_by_metadata'

def _download_safe(symbol: str, period: str, interval: str, prepost: bool = True) -> pd.DataFrame:
    try:
        df = yf.download(
            symbol,
            period=period,
            interval=interval,
            auto_adjust=False,
            progress=False,
            prepost=prepost,
            threads=False,
        )
        return _standardize_ohlcv(df)
    except Exception:
        return pd.DataFrame()

def resolve_symbol(contract_name: str, candidates: list[str], keywords: list[str]) -> tuple[str | None, str, int, str]:
    """Return (resolved_symbol, reason, sample_points, confidence)."""
    first_low_conf = None
    for sym in candidates:
        sample = _download_safe(sym, period='5d', interval='1h', prepost=True)
        if sample.empty:
            continue

        info = _get_symbol_info(sym)
        accepted, reason = _looks_like_target_instrument(sym, info, keywords)
        pts = int(sample['Px'].notna().sum())

        if accepted and reason.startswith('high_confidence'):
            return sym, reason, pts, 'high'

        if accepted and first_low_conf is None:
            first_low_conf = (sym, reason, pts, 'low')

    if first_low_conf is not None:
        return first_low_conf

    return None, 'unresolved_on_yahoo', 0, 'none'

def resolve_all_symbols(contract_specs: dict) -> pd.DataFrame:
    rows = []
    for contract_name, spec in contract_specs.items():
        sym, reason, pts, conf = resolve_symbol(contract_name, spec['candidates'], spec.get('keywords', []))
        rows.append({
            'contract_name': contract_name,
            'contract_code': spec['code'],
            'resolved_symbol': sym,
            'available': sym is not None,
            'sample_points': pts,
            'resolution_reason': reason,
            'resolution_confidence': conf,
        })
    out = pd.DataFrame(rows).sort_values(['available', 'contract_name'], ascending=[False, True])
    out.to_csv(METADATA_FILE, index=False)
    return out

def download_60d_history(symbol: str) -> pd.DataFrame:
    return _download_safe(symbol, period='60d', interval='30m', prepost=True)

def fetch_last_20m_intraday(symbol: str) -> pd.DataFrame:
    # Try finer to coarser intervals, then keep only the last 20m window.
    now = pd.Timestamp.now(tz='UTC')
    for interval in ['1m', '2m', '5m']:
        intr = _download_safe(symbol, period='1d', interval=interval, prepost=True)
        if intr.empty:
            continue

        cutoff = now - pd.Timedelta(minutes=20)
        intr2 = intr[intr.index >= cutoff].copy()
        if not intr2.empty:
            intr2['source_interval'] = interval
            intr2['is_stale_snapshot'] = False
            intr2['age_minutes_at_collection'] = (now - intr2.index).total_seconds() / 60.0
            return intr2

        # market may be closed: keep latest available value as stale snapshot
        last = intr.tail(1).copy()
        if not last.empty:
            last.index = pd.DatetimeIndex([now])
            last['source_interval'] = interval
            last['is_stale_snapshot'] = True
            last['age_minutes_at_collection'] = float((now - intr.index[-1]).total_seconds() / 60.0)
            return last

    return pd.DataFrame()

def append_intraday_dedup(new_rows: pd.DataFrame) -> pd.DataFrame:
    expected = [
        'timestamp_utc', 'contract_name', 'contract_code', 'symbol', 'source_interval',
        'is_stale_snapshot', 'age_minutes_at_collection',
        'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Px'
    ]

    if os.path.exists(INTRADAY_FILE):
        old = pd.read_csv(INTRADAY_FILE, parse_dates=['timestamp_utc'])
    else:
        old = pd.DataFrame(columns=expected)

    if new_rows is None or new_rows.empty:
        return old

    x = new_rows.copy()
    for c in expected:
        if c not in x.columns:
            x[c] = np.nan

    combined = pd.concat([old, x[expected]], ignore_index=True)
    combined['timestamp_utc'] = pd.to_datetime(combined['timestamp_utc'], utc=True, errors='coerce')
    combined = combined.dropna(subset=['timestamp_utc', 'symbol'])
    combined = combined.sort_values('timestamp_utc')
    combined = combined.drop_duplicates(subset=['timestamp_utc', 'contract_code', 'symbol'], keep='last')
    combined.to_csv(INTRADAY_FILE, index=False)
    return combined

def build_price_series(history_df: pd.DataFrame, intraday_df: pd.DataFrame, symbol: str, contract_code: str) -> pd.Series:
    parts = []
    if history_df is not None and not history_df.empty:
        s1 = history_df['Px'].astype(float).dropna()
        s1.index = pd.to_datetime(s1.index, utc=True)
        parts.append(s1)

    if intraday_df is not None and not intraday_df.empty:
        z = intraday_df[(intraday_df['symbol'] == symbol) & (intraday_df['contract_code'] == contract_code)].copy()
        if not z.empty:
            z['timestamp_utc'] = pd.to_datetime(z['timestamp_utc'], utc=True, errors='coerce')
            z = z.dropna(subset=['timestamp_utc', 'Px'])
            s2 = z.set_index('timestamp_utc')['Px'].astype(float).sort_index()
            parts.append(s2)

    if not parts:
        return pd.Series(dtype=float)

    s = pd.concat(parts).sort_index()
    s = s[~s.index.duplicated(keep='last')]
    return s

def asof_value(series: pd.Series, t: pd.Timestamp) -> float:
    if series.empty:
        return np.nan
    sub = series.loc[:t]
    if sub.empty:
        return np.nan
    return float(sub.iloc[-1])

def compute_equal_weight_returns(entity_series: dict[str, pd.Series], horizons: dict[str, pd.Timedelta]) -> pd.DataFrame:
    now = pd.Timestamp.now(tz='UTC')
    rows = []
    for label, delta in horizons.items():
        t0 = now - delta
        rets = []
        for entity, s in entity_series.items():
            p1 = asof_value(s, now)
            p0 = asof_value(s, t0)
            if np.isfinite(p1) and np.isfinite(p0) and p0 != 0:
                rets.append((p1 / p0) - 1.0)
        ew = float(np.mean(rets)) if rets else np.nan
        rows.append({
            'horizon': label,
            'from_utc': t0,
            'asof_utc': now,
            'equal_weight_return': ew,
            'equal_weight_return_pct': ew * 100 if np.isfinite(ew) else np.nan,
            'contributors': len(rets),
        })
    out = pd.DataFrame(rows).sort_values('from_utc')
    out.to_csv(RETURNS_FILE, index=False)
    return out


In [ ]:
# @title 4) Resolve symbols and inspect unresolved contracts
resolved = resolve_all_symbols(CONTRACT_SPECS)
display(resolved[['contract_name','contract_code','resolved_symbol','available','sample_points','resolution_reason','resolution_confidence']])

unresolved = resolved[~resolved['available']]
if not unresolved.empty:
    print('\nUnresolved on Yahoo (kept visible, not replaced with unrelated symbols):')
    display(unresolved[['contract_code', 'contract_name', 'resolution_reason', 'resolution_confidence']])

In [ ]:
# @title 5) Download up-to-60-day history
available = resolved[resolved['available']].copy()
if available.empty:
    raise RuntimeError('No valid Yahoo symbols resolved. Update CONTRACT_SPECS candidates.')

history_cache = {}
for _, row in available.iterrows():
    code = row['contract_code']
    symbol = row['resolved_symbol']
    hist = download_60d_history(symbol)
    history_cache[(code, symbol)] = hist

    out = hist.reset_index().rename(columns={'index': 'timestamp_utc'})
    file_path = os.path.join(HISTORY_DIR, f"{code}__{symbol.replace('=', '_')}_60d_30m.csv")
    out.to_csv(file_path, index=False)
    print(f'Saved 60d history: {code} | {symbol} | rows={len(hist)} -> {file_path}')

In [ ]:
# @title 6) Manual collection run for latest 20 minutes (dedup)
def run_collection_once(resolved_df: pd.DataFrame) -> pd.DataFrame:
    chunks = []
    for _, row in resolved_df[resolved_df['available']].iterrows():
        code = row['contract_code']
        name = row['contract_name']
        symbol = row['resolved_symbol']
        intr = fetch_last_20m_intraday(symbol)
        if intr.empty:
            continue

        z = intr.reset_index().rename(columns={'index': 'timestamp_utc'})
        z['contract_code'] = code
        z['contract_name'] = name
        z['symbol'] = symbol
        chunks.append(z)

    new_rows = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
    intraday_all = append_intraday_dedup(new_rows)
    print(f'Intraday rows in storage after dedup: {len(intraday_all)}')
    return intraday_all

intraday_all = run_collection_once(resolved)
display(intraday_all.tail(20))

In [ ]:
# @title 7) Optional scheduler: run every 15 minutes
RUN_SCHEDULER = False
MAX_RUNS = 8

if RUN_SCHEDULER:
    for i in range(MAX_RUNS):
        print(f'\n=== Scheduled run {i+1}/{MAX_RUNS} @ {datetime.now(timezone.utc).isoformat()} ===')
        _ = run_collection_once(resolved)
        if i < MAX_RUNS - 1:
            time.sleep(15 * 60)

In [ ]:
# @title 8) Build consolidated equal-weight returns
if os.path.exists(INTRADAY_FILE):
    intraday_all = pd.read_csv(INTRADAY_FILE, parse_dates=['timestamp_utc'])
else:
    intraday_all = pd.DataFrame(columns=['timestamp_utc', 'contract_code', 'symbol', 'Px'])

# Build per-contract series
contract_series = {}
symbol_series = {}

for _, row in resolved[resolved['available']].iterrows():
    code = row['contract_code']
    symbol = row['resolved_symbol']

    hist = history_cache.get((code, symbol))
    if hist is None:
        fallback = os.path.join(HISTORY_DIR, f"{code}__{symbol.replace('=', '_')}_60d_30m.csv")
        if os.path.exists(fallback):
            h = pd.read_csv(fallback, parse_dates=['timestamp_utc'])
            h = h.set_index(pd.to_datetime(h['timestamp_utc'], utc=True))
            hist = h
        else:
            hist = pd.DataFrame()

    s = build_price_series(hist, intraday_all, symbol, code)
    if not s.empty:
        contract_series[code] = s
        symbol_series.setdefault(symbol, s)

if COUNT_DUPLICATE_SYMBOLS_MULTIPLE_TIMES:
    series_for_index = contract_series
    mode = 'contract_code_weighting'
else:
    series_for_index = symbol_series
    mode = 'unique_symbol_weighting'

index_returns = compute_equal_weight_returns(series_for_index, HORIZONS)
print('Weighting mode:', mode)
display(index_returns[['horizon', 'equal_weight_return_pct', 'contributors', 'from_utc', 'asof_utc']].sort_values('from_utc'))

In [ ]:
# @title 9) Per-contract transparency table
now = pd.Timestamp.now(tz='UTC')
rows = []
for code, s in contract_series.items():
    p_now = asof_value(s, now)
    row = {'contract_code': code, 'px_now': p_now}
    for k, d in HORIZONS.items():
        p_then = asof_value(s, now - d)
        row[f'{k}_pct'] = (p_now / p_then - 1) * 100 if np.isfinite(p_now) and np.isfinite(p_then) and p_then != 0 else np.nan
    rows.append(row)

per_contract = pd.DataFrame(rows).sort_values('contract_code')
display(per_contract)

## Notes
- `RESOLUTION_MODE='maximize_coverage'` improves coverage when Yahoo lacks clean futures metadata, but rows are labeled with confidence.
- Intraday collector now stores a stale snapshot when markets are closed, so the pipeline still captures a current as-of value.
- By default contracts are equally weighted by requested contract code (`COUNT_DUPLICATE_SYMBOLS_MULTIPLE_TIMES=True`).
